# Load and analyse wildfire data

In [ ]:
# import 

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot
import os
import matplotlib.ticker as ticker
import numpy as np
import xarray as xr
import sys
from utilities import find_best_grid_point, get_station_coords,form_xdate, get_anomalies
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import datetime as dt


from plotting import tol_colors # color schemes from https://personal.sron.nl/~pault/

#activate interactive figures
%matplotlib widget
#activate autoreload
%load_ext autoreload

## Add parent directory to syspath
parent_dir = os.path.abspath(os.path.join(os.path.dirname('.'), '..'))
if not parent_dir in sys.path:
    sys.path.append(parent_dir)

### First, read all our measurement data (GHG) for comparison

In [ ]:
## Read all data
%autoreload 2
from analyses_level2.read_wdc_data import AvailableData, create_data_reader

# File path
data_path = "../data/"

# if New data is added to ./data folder, adapt the dictionary in AvailableData
all_data = list(AvailableData)
print(all_data)

#####---------- TO ADAPT ---------------#####
selected_data = ['CO2', 'CO2_flask', 
                 'CO', 'CO_flask', 
                 'CH4', 'CH4_flask', 
                 'O3'
                 ] # define data to read in. If empty, all data is used 
## 
processing_kwargs = { 
    'FLASK_FLAG_CORR' : True # exclude flagged flask-data
}
#####-----------------------------------#####

datasets = [] # initialize list of all datasets 
# read in data
for sel in (selected_data if selected_data else all_data):
    #define where the data has to be read from
    data_reader =  create_data_reader(data_path=data_path,dataset=sel,**processing_kwargs) #creates an instance of the desired data_reader class
    print(f"Data from {data_reader.__class__.__name__} for {sel}:")

    # call the data-reading function on that instance: 
    data = data_reader.read_data() 
    # call the data-processing
    data = data_reader.process_data(data)

    # prepare merged dataset
    data = data.drop(columns='endtime') # problem when merging datasets (because of NaT?), so better remove endtime
    ds = data.to_xarray()
    ds = ds.assign_coords(dataset=sel)
    ds['species'] = data_reader.species
    ds['unit']  = np.unique(ds.unit.dropna(dim='time'))[0]
    datasets.append(ds)

# save all in one xarray dataset
ds_all = xr.concat(datasets,dim="dataset")
ds_all

### check GFAS fire data

In [ ]:
## Read in CAMS GFAS fire data
# Read all full cams datasets
dir_data_cams = r"../data/cams"
cams_gfas = xr.open_dataset(dir_data_cams + r"/cams_gfas_2020_2023.nc")

In [ ]:
##Take a mean of the fire activity over the whole area
t1 = "2020-01-01"
t2 = "2023-12-31"
gfas_mean = cams_gfas.mean(dim="latitude").mean(dim="longitude").sel(time=slice(t1, t2))

# define which measurement I would like to plot
obs_var = "CO"

In [ ]:
## select a more local fire activity mean

mknlat, mknlon, mknalt = get_station_coords("MKN")  # Mt. Kenya station coordinates

# define a "local" area
dlat = 5  # lat difference to station
dlon = 5 # lon difference to station
[lat_min, lat_max, lon_min, lon_max] = [
    mknlat - dlat,
    mknlat + dlat,
    mknlon - dlon,
    mknlon + dlon,
]

gfas_mean_local = cams_gfas.sel(latitude=slice(lat_max,lat_min), longitude = slice(lon_min,lon_max)).mean(dim="latitude").mean(dim="longitude").sel(time=slice(t1, t2))

In [ ]:
## plot the fireactivity as a time series
# If desired, distinguish between north and south of MKN station

fig, ax = plt.subplots(1, 1)
t1 = "2020-01-01"
t2 = "2023-12-31"

gfas_mean["frpfire"].plot(label="Total mean fire activity",color='C1')
gfas_mean_local["frpfire"].plot(label="Local fire activity",color='C3',ls=':')

# to distinguish between north and south:
# cams_gfas.where(cams_gfas.latitude > mknlat).mean(dim='latitude').mean(dim='longitude').sel(time=slice(t1,t2))['frpfire'].plot(label='north')
# cams_gfas.where(cams_gfas.latitude < mknlat).mean(dim='latitude').mean(dim='longitude').sel(time=slice(t1,t2))['frpfire'].plot(label='south')
ax2 = ax.twinx()

## plot measurement data
ds_all.sel(dataset=obs_var)["value"].sel(time=slice(t1, t2)).resample(
    time="1D"
).mean().plot(ax=ax2, label="CO", color="C0")
# ax2.set_ylim([50,300])
ax2.set_ylabel(obs_var)
# cams_gfas.mean(dim='latitude').mean(dim='longitude')['cofire'].sel(time=slice(t1,t2)).plot(ax=ax2,color='red') #cams CO (not working?)
plt.title("Wildfire radiative power in large area")
plt.tight_layout()
ax.legend(loc="upper left")
ax2.legend(loc="upper right")
plt.savefig(f"./output/fire_activity/timeseries_firec_co_local_{dlat*2}x{dlon*2}deg.pdf")
#plt.savefig("./output/fire_activity/timeseries_firec_co_NS.pdf")
plt.show()

In [ ]:

from matplotlib.colors import LogNorm, SymLogNorm

## Show a map with all fire activity for a specific period

t1 = "2020-01-01"
t2 = "2023-12-31"

gfas_sel = cams_gfas["frpfire"].sel(time=slice(t1, t2)).mean(dim="time")
#only more local fires: 
#gfas_sel = cams_gfas.sel(latitude=slice(lat_max,lat_min), longitude = slice(lon_min,lon_max))["frpfire"].sel(time=slice(t1, t2)).mean(dim="time")
gfas_sel = gfas_sel.where(
    gfas_sel != 0, other=np.nan
)  # only plot non-zero values, set all others to nan

# area to map
[lon1, lon2, lat1, lat2] = [0, 60, -40, 40]

# start the figure
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
plt.gca().set_extent([lon1, lon2, lat1, lat2], crs=ccrs.PlateCarree())

lands = cfeature.NaturalEarthFeature(
    category="physical", name="land", scale="50m"
)  # cfeature.COLORS['land']
ax.add_feature(lands, zorder=0)
ax.add_feature(cfeature.BORDERS, linestyle=":", edgecolor="black")

gfas_sel.plot(
    cmap="hot",
    norm = LogNorm(),
    transform=ccrs.PlateCarree(),
    #add_colorbar = True,
    cbar_kwargs={"label": 'Fire Radiative Power (W/m^2)'}
)

# or: do the log manually for the map: 
#plt.pcolormesh(gfas_sel["longitude"],gfas_sel["latitude"],-np.log10(gfas_sel),cmap="hot",transform=ccrs.PlateCarree())
#cbar = plt.colorbar(label="-log(Fire Radiative Power [W/m^2])")
#cbar.set_ticks(np.arange(np.floor(np.min(ds_toplot)), np.ceil(np.max(ds_toplot)) + 1))

#adapt the colorbar

mknlat, mknlon, mknalt = get_station_coords("MKN")
ax.plot(
    mknlon, mknlat, marker="o", color="k", transform=ccrs.PlateCarree(), zorder=2
)  # mt. kenya station

plt.show()

### Do some scatterplots fireactivity vs. CO

In [ ]:
from sklearn.linear_model import LinearRegression

## Mean fire vs. CO for different months


# x = cams_gfas.sel(time=slice(t1, t2)).mean(dim=["latitude", "longitude"])["frpfire"]
x = gfas_mean["frpfire"]
# ,latitude=slice(5,-5),longitude=slice(32,42)
y = ds_all.sel(dataset=obs_var, time=slice(t1, t2))["value"].resample(time="D").mean()
months = x.time.dt.month

# Define a colormap for months
colmap = tol_colors.tol_cmap("rainbow_discrete", 12)  # 12 colors for 12 months
colors = colmap(np.linspace(0, 1, 12))

fig, axs = plt.subplots(1,3,figsize=(15,4),sharey=True,sharex=True)
for i,ax in enumerate(axs): 
    #first subplot: all months scatter
    if i == 0: 
        scatter = ax.scatter(x, y, c=months, cmap=colmap, alpha=1)    
        legend1 = ax.legend(*scatter.legend_elements(), loc="upper right", title="Months")
        ax.add_artist(legend1)

    #second subplot: all months scatter with linear fits
    elif i == 1: 
        scatter = ax.scatter(x, y, c=months, cmap=colmap, alpha=1)
        #legend1 = ax.legend(*scatter.legend_elements(), loc="upper right", title="Months")
        #ax.add_artist(legend1)
        ## add line fits:
        # Iterate over unique months and fit a line for each month
        for month, c in zip(range(1, 13), colors):
            # Filter data for the current month
            x_month = x.where(x.time.dt.month == month, drop=True)
            y_month = y.where(y.time.dt.month == month, drop=True)

            # CO data contains some nans, remove them to be able to make a fit
            y_month_non_nan = y_month.dropna(dim="time")
            x_month_non_nan = x_month.sel(time=y_month_non_nan.time.values, drop=True)

            if len(x_month) > 1:
                # Fit a linear regression model
                model = LinearRegression()
                model.fit(x_month_non_nan.values.reshape(-1, 1), y_month_non_nan.values)

                # Predictions
                x_pred = np.linspace(
                    x.min(), x.max(), 2
                )  # create x-values for which to predict y-values
                y_pred = model.predict(x_pred.reshape(-1, 1))

                # Plot the line fit for the current month
                ax.plot(
                    x_pred, y_pred, color=c, label=f"Month {month}"
                )  # color=colmap(month / 13)
                # ax.plot(x_pred, y_pred, color=colmap(month / 13), label=f'Month {month}')
            else:
                print(f"no fit for month {month}")

    # 3rd subplot: only single month
    elif i ==2: 
        month = 3 ## Select a specific month
        x_month = x.where(x.time.dt.month == month, drop=True)
        y_month = y.where(y.time.dt.month == month, drop=True)


        scatter = ax.scatter(x_month, y_month, alpha=1, c=colors[month - 1])
        dt.datetime.strptime('1', '%m').strftime("%b")
        str_mon = dt.datetime.strptime(str(month), '%m').strftime("%B")
        ax.set_title(str_mon)    
    
        # Fit a linear regression model
        # CO data contains some nans, remove them to be able to make a fit
        y_month_non_nan = y_month.dropna(dim="time")
        x_month_non_nan = x_month.sel(time=y_month_non_nan.time.values, drop=True)
        if len(x_month) > 1:
            model = LinearRegression()
            model.fit(x_month_non_nan.values.reshape(-1, 1), y_month_non_nan.values)

            # Predictions
            x_pred = np.linspace(x.min(), x.max(), 10) # using x (instead of x_month) to have same x values as for all months
            y_pred = model.predict(x_pred.reshape(-1, 1))

            # Plot the line fit for the current month
            ax.plot(x_pred, y_pred, color=colors[month - 1], label=f"Month {month}")


    ax.set_ylabel(f"{obs_var} at MKN")
    ax.set_xlabel("Mean fire activity (radiative power in W/m2)")

plt.suptitle('Fire events (whole central+eastern Africa) vs. CO at Mt. Kenya for different months')
plt.savefig(f"./output/fire_activity/scatter_plots_fire_{obs_var}.pdf")
plt.show()

In [ ]:
## distinguish fire events north and south of mount kenya station
gfas_north = cams_gfas.where(cams_gfas.latitude > mknlat)
gfas_south = cams_gfas.where(cams_gfas.latitude < mknlat)

In [ ]:
# same (fire activity vs. CO) but for north and south from MKN station
mknlat, mknlon, mknalt = get_station_coords("MKN")

# TO ADAPT
line_fit = False

fig, axs = plt.subplots(1, 2, figsize=(10, 6), sharey=True, sharex=True)
## devide in different latitude regions (above and below the station)
for i, ax in enumerate(axs):
    y = ds_all.sel(dataset="CO", time=slice(t1, t2))["value"].resample(time="D").mean()

    # select fires above or below the station
    if i == 0:
        gfas_sel = gfas_north
        tit = "Fire events north of MKN"
    else:
        gfas_sel = gfas_south
        tit = "Fire events south of MKN"

    x = gfas_sel.sel(time=slice(t1, t2)).mean(dim=["latitude", "longitude"])["frpfire"]
    months = x.time.dt.month
    # Define a colormap for months
    colmap = tol_colors.tol_cmap("rainbow_discrete", 12)  # 12 colors for 12 months
    scatter = ax.scatter(
        x,
        y,
        c=months,
        cmap=colmap,
    )
    ax.set_xlabel("Mean fire activity (radiative power in W/m2)")
    ax.set_title(tit)

    if line_fit: 
        ## add line fits:
        # Iterate over unique months and fit a line for each month
        for month, c in zip(range(1, 13), colors):
            # Filter data for the current month
            x_month = x.where(x.time.dt.month == month, drop=True)
            y_month = y.where(y.time.dt.month == month, drop=True)

            # CO data contains some nans, remove them to be able to make a fit
            y_month_non_nan = y_month.dropna(dim="time")
            x_month_non_nan = x_month.sel(time=y_month_non_nan.time.values, drop=True)

            if len(x_month) > 1:
                # Fit a linear regression model
                model = LinearRegression()
                model.fit(x_month_non_nan.values.reshape(-1, 1), y_month_non_nan.values)

                # Predictions
                x_pred = np.linspace(
                    x.min(), x.max(), 2
                )  # create x-values for which to predict y-values
                y_pred = model.predict(x_pred.reshape(-1, 1))

                # Plot the line fit for the current month
                ax.plot(
                    x_pred, y_pred, color=c, label=f"Month {month}"
                )  # color=colmap(month / 13)
                # ax.plot(x_pred, y_pred, color=colmap(month / 13), label=f'Month {month}')
            else:
                print(f"no fit for month {month}")


plt.legend()
axs[0].set_ylabel("CO at MKN")
# plt.colorbar()
legend1 = ax.legend(*scatter.legend_elements(), loc="upper right", title="Months")
ax.add_artist(legend1)
plt.suptitle(f"Mean regional fire events (CAMS) vs. CO at MKN between {t1} and {t2}")



#plt.savefig("./output/fire_activity/scatter_firec_co_NS.pdf")
# plt.savefig('.\output\scatter_firec_co_EW.pdf')
plt.show()

In [ ]:
### Weight the fire activity with the distance to MKN station
dx = cams_gfas["longitude"] - mknlon
dy = cams_gfas["latitude"] - mknlat
dist = np.sqrt(dx**2 + dy**2)
cams_gfas["dist_mkn"] = dist
fire_weighted = cams_gfas["frpfire"] * (1 / dist)
cams_gfas["frpfire_weighted"] = fire_weighted

# TODO would need to make dist_mkn a coordinate, and make frpfire dependent on that, so that we have a dist value for each fire event!?

In [ ]:
## Check fire activity around MKN for a specific period

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

d1 = "2022-03-20"
d2 = "2022-03-28"
var = "frpfire"
ds_sel = cams_gfas.sel(time=slice(d1, d2))

fig, axs = plt.subplots(3, 3, sharex=True, sharey=True, figsize=(8, 8))

# area to map
[lon1, lon2, lat1, lat2] = [36, 39, -1, 1] # close zoom to MKN station

for d, ax in zip(pd.date_range(start=d1, end=d2), axs.reshape(-1)):
    vmin = ds_sel[var].min()
    vmax = ds_sel[var].max()
    im = cams_gfas.sel(time=d)[var].plot(
        ax=ax, vmin=vmin, vmax=vmax, add_colorbar=False,cmap='hot_r'
    )
    ax.plot(mknlon, mknlat, marker="x", color="g")  # mt. kenya station
    ax.set_xlabel("lon")
    ax.set_ylabel("lat")
# common colorbar
fig.colorbar(
    im,
    ax=axs.ravel().tolist(),
    orientation="vertical",
    label="Wildfire radiative power (W/m2)",
    pad=-0.5,
    #norm = LogNorm()
)
ax.set_xlim([lon1,lon2])
ax.set_ylim([lat1,lat2])
plt.tight_layout()
plt.show()

### Check Seasonal and daily cycles

In [ ]:
## plot  SEASONAL CYCLE fires (cams) versus CO observations

alpha = 0.7
ms = 8

# use cams gfas
t1 = "2020-01-01"
t2 = "2023-12-31"

species_sel = ["CO"]

# resample measurements to 1 day
meas_1D = (
    ds_all.sel(dataset=species_sel, time=slice(t1, t2))["value"]
    .resample(time="D")
    .mean()
)

freq = "month"
fig, ax = plt.subplots(1, 1, sharex=True, figsize=(8,5))

# function to plot each subplot
def plot_data(ds, ax, label,c,ls='-'):
    l = ds.groupby(f"time.{freq}").mean().plot(
        ax=ax,
        ls=ls,
        c=c,
        marker=".",
        markeredgewidth=0,
        alpha=alpha,
        markersize=ms,
        label=f"{label}",
    )
    return l


# plot cams
fire_color = 'C3' # the new matplotlib default colors can be accessed with CN (N=0-9)
l1 = plot_data(gfas_mean.sel(time=slice(t1, t2))["frpfire"], ax, "fire activity",fire_color)
l3 = plot_data(gfas_mean_local.sel(time=slice(t1, t2))["frpfire"], ax, f"Local fire activity +/-{dlat}°",fire_color,ls=':')
ax2.set_ylabel('Mean fire activity (W/m2)')

ax2 = ax.twinx()
# plot obs
meas_color = 'C0'
l2 = plot_data(meas_1D, ax2, "CO MKN",meas_color)
ax2.set_ylabel('CO (ppb)')
#set axes to the measurement color
ax2.yaxis.label.set_color(meas_color)
ax2.spines['right'].set_color(meas_color)
ax2.tick_params(axis='y', colors=meas_color)

#common legend
lns = l1+l2+l3
labs = [l.get_label() for l in lns]
fig.legend(lns, labs)


# ax.set_ylabel(f"")
ax.set_xlabel("")
ax.set_title("")

if freq == "month":
    tit = "Seasonal"
elif freq == "hour":
    tit = "Diurnal"

plt.suptitle(f"{tit} cycle")
plt.tight_layout()
plt.savefig(f"./output/fire_activity/seasonal_firec_co_local_{dlat*2}x{dlon*2}deg.pdf")
plt.show()